# Notebook 1 — Exploração de Dados e Análise de Drift em Escala

## Aula 7: Otimizações e Escala no Monitoramento de Drift

### Objetivos

1. Carregar e explorar o dataset sintético **FinBank Transactions** (10.000 transações).
2. Analisar distribuições temporais das features por janela mensal (meses 1–8).
3. Visualizar drift gradual (meses 4–6) e abrupto (meses 7–8) injetado no dataset.
4. Calcular PSI e K–S para cada feature, comparando baseline vs. janelas correntes.
5. Compreender os desafios de Big Data: volume, velocidade e custo computacional.

### Conexão com o Documento 04

> *"Imagine treinar um modelo de detecção de fraudes bancárias que funciona muito bem hoje,
> mas que amanhã começa a falhar porque o perfil das transações mudou — e ninguém percebeu
> a tempo. Esse é o desafio do data drift em escala."*
> — DOCUMENTO_AULA_7.md, seção 'O Que Vem Por Aí?'

### Vídeo Relacionado

- **Vídeo 7.1**: Caso FinBank (fraude em escala massiva); desafios de Big Data;
  limitações do monitoramento manual em batch.
- **Vídeo 7.2**: Arquiteturas Lambda e Kappa para streaming de drift.

In [ ]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp

# Adicionar diretório pai ao path para importar src/
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor
from src.model import DriftMonitor

# Configurações de visualização
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

%matplotlib inline

## 1. Carregamento dos Dados FinBank

O dataset simula o cenário do **caso FinBank**: uma fintech com alto volume de transações
financeiras. O dataset contém **10.000 transações** distribuídas ao longo de **8 meses**,
com drift injetado conforme o seguinte padrão:

| Período | Meses | Comportamento |
|---------|-------|---------------|
| Baseline | 1–3 | Distribuição estável |
| Drift gradual | 4–6 | Aumento em amount, mudança em channel |
| Drift abrupto | 7–8 | Mudança forte em múltiplas features |

Conforme a seção 'Saiba Mais' do DOCUMENTO_AULA_7.md:
> *"Drift descreve alterações estatísticas na distribuição dos dados de entrada,
> na relação entre atributos e rótulos, ou em ambas as coisas simultaneamente."*

In [ ]:
# Carregar dataset FinBank
preprocessor = DataPreprocessor(random_state=42)

try:
    df = preprocessor.load_data("../data/raw/finbank_transactions.csv")
except FileNotFoundError:
    print("Dataset não encontrado. Execute primeiro:")
    print("  python scripts/generate_dataset.py")
    raise

print(f"Shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")
df.head()

## 2. Estatísticas Descritivas por Mês

Analisamos a evolução temporal das features para identificar visualmente
possíveis padrões de drift. Como descrito na Videoaula 1, o monitoramento
manual em batch tem limitações severas em cenários de Big Data.

In [ ]:
# Distribuição de amostras por mês e taxa de fraude
print("Amostras por mês:")
print(df["month"].value_counts().sort_index())
print("\nTaxa de fraude por mês:")
print(df.groupby("month")["is_fraud"].mean().round(4))

In [ ]:
# Estatísticas descritivas das features numéricas
numeric_features = DataPreprocessor.NUMERIC_FEATURES
print(f"Features numéricas monitoradas: {numeric_features}\n")
df[numeric_features].describe().round(2)

## 3. Separação Baseline vs. Corrente

Conforme discutido na seção 'Saiba Mais' — janelas, amostragem e trade-offs de custo:
> *"Medir drift significa comparar duas distribuições: uma distribuição de referência
> (tipicamente o conjunto de treinamento, ou um período estável do passado) e uma
> distribuição corrente."*

Usamos os meses 1–3 como baseline estável.

In [ ]:
# Separar baseline (meses 1-3) vs. corrente (meses 4-8)
splits = preprocessor.split_data(df, baseline_months=(1, 2, 3))
baseline = splits["baseline"]
current = splits["current"]

print(f"Baseline (meses 1-3): {len(baseline)} transações")
print(f"Corrente (meses 4-8): {len(current)} transações")
print(f"\nFraude baseline: {baseline['is_fraud'].mean():.2%}")
print(f"Fraude corrente: {current['is_fraud'].mean():.2%}")

## 4. Visualização de Drift por Feature

Comparamos as distribuições de cada feature numérica entre baseline e os
diferentes períodos (drift gradual, drift abrupto).

Conforme o Quadro 1 do DOCUMENTO_AULA_7.md, o PSI mede divergência por bins
(interpretação simples), enquanto o K–S fornece evidência estatística formal.

In [ ]:
# Visualizar distribuições: baseline vs. drift gradual vs. drift abrupto
df_baseline = df[df["month"].isin([1, 2, 3])]
df_gradual = df[df["month"].isin([4, 5, 6])]
df_abrupto = df[df["month"].isin([7, 8])]

key_features = ["amount", "customer_age", "transaction_count_30d", "avg_amount_30d"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.ravel(), key_features):
    ax.hist(df_baseline[feat], bins=30, alpha=0.4, density=True,
            label="Baseline (1-3)", color="steelblue", edgecolor="black", linewidth=0.5)
    ax.hist(df_gradual[feat], bins=30, alpha=0.4, density=True,
            label="Gradual (4-6)", color="orange", edgecolor="black", linewidth=0.5)
    ax.hist(df_abrupto[feat], bins=30, alpha=0.4, density=True,
            label="Abrupto (7-8)", color="red", edgecolor="black", linewidth=0.5)
    ax.set_title(f"Distribuição: {feat}")
    ax.set_xlabel(feat)
    ax.set_ylabel("Densidade")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Comparação de Distribuições: Baseline vs. Drift Gradual vs. Abrupto",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/distribuicoes_drift.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Análise Mensal de Drift — PSI e K–S

Calculamos o PSI e o teste K–S para cada feature, mês a mês.

O PSI é definido como (Kullback & Leibler, 1951):

$$PSI = \sum_{i=1}^{B} (q_i - p_i)\,\ln\left(\frac{q_i}{p_i}\right)$$

E o teste K–S compara CDFs empíricas (Kolmogorov, 1933; Smirnov, 1948):

$$D_{n,m} = \sup_x |F_n(x) - G_m(x)|$$

Interpretação do PSI:
- PSI < 0.10 → Sem mudança significativa
- 0.10 ≤ PSI < 0.25 → Mudança moderada (warning)
- PSI ≥ 0.25 → Mudança severa (critical)

In [ ]:
# Calcular PSI e K–S por feature e por mês
# Implementa Snippet 2 e 3 do Hands On
monitor = DriftMonitor(n_bins=10, psi_threshold=0.25, psi_warning=0.10, ks_alpha=0.05)

windows = preprocessor.get_monthly_windows(df)
baseline_data = {feat: df_baseline[feat].values for feat in numeric_features}

# Tabela de resultados
results_table = []

for month in sorted(windows.keys()):
    if month <= 3:
        continue  # Pular baseline
    window_df = windows[month]
    for feat in numeric_features:
        ref = baseline_data[feat]
        cur = window_df[feat].values
        psi_val = monitor.compute_psi(ref, cur)
        ks_stat, ks_pval = monitor.compute_ks(ref, cur)
        wass = monitor.compute_wasserstein(ref, cur)

        severity = "critical" if psi_val >= 0.25 else "warning" if psi_val >= 0.10 else "ok"

        results_table.append({
            "Mês": month,
            "Feature": feat,
            "PSI": round(psi_val, 4),
            "KS_stat": round(ks_stat, 4),
            "KS_pvalue": round(ks_pval, 4),
            "Wasserstein": round(wass, 2),
            "Severidade": severity,
        })

results_df = pd.DataFrame(results_table)
print("Métricas de Drift — Baseline (meses 1-3) vs. Janela Mensal")
results_df

## 6. Heatmap de PSI por Feature e Mês

Visualização consolidada que permite identificar quais features
e em que momento o drift se intensifica. Conforme o DOCUMENTO_AULA_7.md:

> *"Em sistemas distribuídos, o uso de estados agregados (contagens por bin,
> somas, quantis aproximados) permite computar PSI e outras métricas sem
> materializar o conjunto completo de dados."*

In [ ]:
# Heatmap de PSI por feature e mês
pivot = results_df.pivot(index="Feature", columns="Mês", values="PSI")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
            linewidths=0.5, cbar_kws={"label": "PSI"})
ax.set_title("Heatmap de PSI por Feature e Mês\n(Baseline: meses 1–3)", fontsize=13)
ax.set_xlabel("Mês")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.savefig("../outputs/figures/heatmap_psi.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Evolução Temporal do PSI Médio

Observamos a tendência do PSI médio ao longo do tempo, evidenciando a
progressão de drift gradual (meses 4–6) para drift abrupto (meses 7–8).
Essa visualização conecta-se ao conceito de playbooks de drift:

> *"Se PSI excede um limiar em uma feature crítica por k janelas consecutivas,
> abre-se incidente e inicia-se investigação."*
> — DOCUMENTO_AULA_7.md, seção 'Drift, observabilidade e governança'

In [ ]:
# Evolução temporal do PSI médio
psi_by_month = results_df.groupby("Mês")["PSI"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["red" if v >= 0.25 else "orange" if v >= 0.10 else "green"
          for v in psi_by_month.values]

ax.bar(psi_by_month.index, psi_by_month.values, color=colors, alpha=0.8,
       edgecolor="black", linewidth=0.5)
ax.axhline(y=0.10, color="orange", linestyle="--", label="Warning (0.10)")
ax.axhline(y=0.25, color="red", linestyle="--", label="Crítico (0.25)")
ax.set_xlabel("Mês")
ax.set_ylabel("PSI Médio")
ax.set_title("Evolução Temporal do PSI Médio (Baseline: meses 1–3)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/psi_temporal.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Comparação de CDFs — Teste K–S Visual

Conforme a seção 'Saiba Mais', o teste K–S compara CDFs empíricas.
Visualizamos a estatística $D_{n,m}$ como a maior diferença vertical
entre as duas curvas (Kolmogorov, 1933; Smirnov, 1948).

In [ ]:
# Comparação visual de CDFs: amount (baseline vs. mês 8)
from src.evaluation import plot_distribution_comparison

ref_amount = df_baseline["amount"].values
cur_amount = df[df["month"] == 8]["amount"].values

fig = plot_distribution_comparison(
    ref_amount, cur_amount,
    feature_name="amount (R$)",
    save_path="../outputs/figures/cdf_amount_m8.png",
)

# Calcular e exibir K–S
ks_stat, ks_pval = ks_2samp(ref_amount, cur_amount)
print(f"K–S Statistic: {ks_stat:.4f}")
print(f"K–S p-value:   {ks_pval:.2e}")
print(f"Drift detectado: {'Sim' if ks_pval < 0.05 else 'Não'}")
plt.show()

## 9. Distribuição do Canal de Transação (Feature Categórica)

Além das features numéricas, monitoramos mudanças na distribuição
de variáveis categóricas. No caso FinBank, o canal de transação
muda significativamente nos meses de drift abrupto (mais transações
por app e phone).

In [ ]:
# Distribuição do canal por período
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (label, subset) in zip(axes, [
    ("Baseline (1-3)", df_baseline),
    ("Gradual (4-6)", df_gradual),
    ("Abrupto (7-8)", df_abrupto),
]):
    counts = subset["channel"].value_counts(normalize=True).sort_index()
    ax.bar(counts.index, counts.values, color="steelblue", alpha=0.7,
           edgecolor="black", linewidth=0.5)
    ax.set_title(label)
    ax.set_ylabel("Proporção")
    ax.set_ylim(0, 0.7)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Distribuição do Canal de Transação por Período", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/channel_drift.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Desafios de Big Data e Amostragem

Conforme discutido na Videoaula 1 e na seção 'Saiba Mais':
> *"Em Big Data, o problema não é apenas detectar drift, mas detectá-lo sob
> restrições severas: alta taxa de eventos, baixa latência e custo computacional
> controlado."*

Demonstramos o efeito da amostragem estratificada na qualidade da estimativa de PSI.

In [ ]:
# Efeito da amostragem na estimativa de PSI
# Simula trade-off custo vs. qualidade estatística
fractions = [0.05, 0.10, 0.20, 0.50, 1.0]
feat_test = "amount"
ref = df_baseline[feat_test].values
full_current = df[df["month"] == 7][feat_test].values

psi_full = monitor.compute_psi(ref, full_current)

psi_results = []
n_trials = 20

for frac in fractions:
    trial_psis = []
    for trial in range(n_trials):
        rng = np.random.RandomState(trial)
        n_sample = max(int(len(full_current) * frac), 10)
        sample = rng.choice(full_current, size=n_sample, replace=False)
        trial_psis.append(monitor.compute_psi(ref, sample))
    psi_results.append({
        "Fração": frac,
        "n_amostras": int(len(full_current) * frac),
        "PSI_medio": np.mean(trial_psis),
        "PSI_std": np.std(trial_psis),
        "Erro_relativo_%": abs(np.mean(trial_psis) - psi_full) / psi_full * 100,
    })

psi_sample_df = pd.DataFrame(psi_results)
print(f"PSI com 100% dos dados (mês 7 vs baseline): {psi_full:.4f}\n")
psi_sample_df

In [ ]:
# Visualização: erro relativo vs. fração de amostragem
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([r["Fração"] for r in psi_results],
        [r["Erro_relativo_%"] for r in psi_results],
        marker="o", color="steelblue", linewidth=2, markersize=8)
ax.set_xlabel("Fração de Amostragem")
ax.set_ylabel("Erro Relativo no PSI (%)")
ax.set_title("Trade-off Amostragem vs. Precisão do PSI (feature: amount)")
ax.grid(alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("../outputs/figures/amostragem_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

## Resumo

Neste notebook:

1. **Exploramos** o dataset FinBank com 10.000 transações distribuídas em 8 meses.
2. **Visualizamos** drift gradual (meses 4–6) e abrupto (meses 7–8) nas features.
3. **Calculamos** PSI, K–S e Wasserstein para cada feature e mês.
4. **Analisamos** o trade-off entre amostragem e precisão da estimativa de PSI.

### Próximo Notebook

No **Notebook 02 — Treinamento**, implementaremos o pipeline de janelas deslizantes
com monitoramento contínuo de drift, incluindo cálculo incremental de PSI e simulação
de streaming em lote.